In [ ]:

import os
import glob
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import gc

In [ ]:

# =========================================================
# 1. 路径设置
# =========================================================

main_folder = r"E:\SHARE\E10006\DATAFILE\diagnosis"

output_folder = r"E:\SHARE\E10006\PARQUET"

# 如果输出文件夹不存在，则创建
os.makedirs(output_folder, exist_ok=True)


# =========================================================
# 2. 获取 diagnosis 下所有 dataset 子文件夹
# =========================================================

dataset_folders = sorted([
    folder
    for folder in os.listdir(main_folder)
    if os.path.isdir(os.path.join(main_folder, folder))
])

print("Dataset amount:", len(dataset_folders))


# =========================================================
# 3. 逐个 dataset 处理
# =========================================================

for dataset_name in dataset_folders:

    dataset_path = os.path.join(
        main_folder,
        dataset_name
    )

    # 找这个 dataset 文件夹下所有 gz 文件
    gz_files = sorted(
        glob.glob(
            os.path.join(dataset_path, "*.gz")
        )
    )

    # 如果没有 gz，跳过
    if len(gz_files) == 0:
        print(f"\n skip {dataset_name}：no gz files found")
        continue


    # 输出 parquet 文件
    output_file = os.path.join(
        output_folder,
        f"{dataset_name}.parquet"
    )

    if os.path.exists(output_file):
    print(f"\n skip {dataset_name}：Parquet have already been generated")
    continue


    print("\n======================================")
    print("Dataset:", dataset_name)
    print("GZ file amount:", len(gz_files))
    print("Output file:", output_file)
    print("======================================")


    writer = None
    total_rows = 0


    try:

        # ============================================
        # 4. 读取该 dataset 下所有 gz
        # ============================================

        for gz_file in gz_files:

            print("\nReading:", os.path.basename(gz_file))

            # 分块读取，避免占用过多内存
            for chunk in pd.read_csv(
                gz_file,
                compression="gzip",
                sep="\x01",
                chunksize=500000,
                low_memory=False
            ):

                # pandas -> Arrow
                table = pa.Table.from_pandas(
                    chunk,
                    preserve_index=False
                )


                # 第一个 chunk 创建 parquet writer
                if writer is None:

                    writer = pq.ParquetWriter(
                        output_file,
                        table.schema,
                        compression="zstd"
                    )


                # 写入 parquet
                writer.write_table(table)


                total_rows += len(chunk)

                print(
                    "累计写入:",
                    f"{total_rows:,}",
                    "rows"
                )


                # 清理当前 chunk
                del chunk
                del table
                gc.collect()


    finally:

        # ============================================
        # 5. 当前 dataset 写完后关闭文件
        # ============================================

        if writer is not None:
            writer.close()


    # ================================================
    # 6. 检查输出文件
    # ================================================

    if os.path.exists(output_file):

        size_mb = (
            os.path.getsize(output_file)
            / 1024**2
        )

        print("\n完成:", dataset_name)
        print("Total rows:", f"{total_rows:,}")
        print("Parquet size:", round(size_mb, 2), "MB")

    else:

        print("\n失败:", dataset_name)
        print("No parquet file generated")


print("\nAll dataset processing completed!")

In [ ]:
# check repeated columns

import os
import glob
import pyarrow.parquet as pq
from collections import defaultdict

parquet_folder = r"E:\SHARE\E10006\PARQUET"

parquet_files = glob.glob(
    os.path.join(parquet_folder, "*.parquet")
)

column_sources = defaultdict(list)

for file in parquet_files:

    file_name = os.path.basename(file)

    schema = pq.read_schema(file)

    for col in schema.names:
        column_sources[col].append(file_name)


# 只显示存在于两个及以上数据集的变量
for col, files in sorted(column_sources.items()):

    if len(files) > 1:

        print("\n变量：", col)

        for f in files:
            print("   ", f)

In [ ]:

dataset_files = {

    "D01": "patient_data.parquet",
    "D02": "dmcs_data.parquet",
    "D03": "fm_pat_disease.parquet",
    "D04": "fm_cn_problem.parquet",
    "D06": "mdl_dx_progress.parquet",

    "D08": "opas_case_data.parquet",
    "D09": "aeis_case_data.parquet",
    "D12": "ipas_case_data1.parquet"
}

In [ ]:
#read D08
import duckdb

con = duckdb.connect()

folder = "E:/SHARE/E10006/PARQUET"

con.execute(f"""
CREATE VIEW D08 AS
SELECT *
FROM read_parquet('{folder}/opas_case_data.parquet')
""")

con.execute(f"""
CREATE VIEW D06 AS
SELECT *
FROM read_parquet('{folder}/mdl_dx_progress.parquet')
""")

# check columns in D06
d06_cols = [
    row[0]
    for row in con.execute(
        "DESCRIBE D06"
    ).fetchall()
]

d06_cols

# create SELECT statement for D06s
d06_select = ",\n".join(

    [
        f'd6."{col}" AS "D06_{col}"'

        for col in d06_cols

        if col not in [
            "pseudo_patient_key",
            "pseudo_episode_key"
        ]
    ]
)

print(d06_select)

# join D08 and D06
sql = f"""

SELECT

    d08.*,

    {d06_select}

FROM D08 d08

LEFT JOIN D06 d6

    ON d08.pseudo_patient_key =
       d6.pseudo_patient_key

   AND d08.pseudo_episode_key =
       d6.pseudo_episode_key

   AND d6.src_case_type = 'O'

"""